In [ ]:
%env CUDA_VISIBLE_DEVICES=0
import os
os.environ["CUDA_VISIBLE_DEVICES"]="0"
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

import argparse
import torch

from llava.constants import IMAGE_TOKEN_INDEX, DEFAULT_IMAGE_TOKEN, DEFAULT_IM_START_TOKEN, DEFAULT_IM_END_TOKEN
from llava.conversation import conv_templates, SeparatorStyle
from llava.model.builder import load_pretrained_model
from llava.utils import disable_torch_init
from llava.mm_utils import process_images, tokenizer_image_token, get_model_name_from_path
from transformers.generation.utils import GenerationMixin

from PIL import Image
import torch.nn.functional as F

import pandas as pd
import requests
from PIL import Image
from io import BytesIO
from transformers import TextStreamer
import numpy as np
import sys


def load_image(image_file):
    if image_file.startswith('http://') or image_file.startswith('https://'):
        response = requests.get(image_file)
        image = Image.open(BytesIO(response.content)).convert('RGB')
    else:
        image = Image.open(image_file).convert('RGB')
    return image

if "ipykernel_launcher" in sys.argv[0]:
    sys.argv = sys.argv[:1]

parser = argparse.ArgumentParser()

parser.add_argument("--model-path", type=str, default="liuhaotian/llava-v1.5-7b")
parser.add_argument("--model-base", type=str, default=None)
parser.add_argument("--image-file", type=str, default=None)
parser.add_argument("--device", type=str, default="cuda")
parser.add_argument("--conv-mode", type=str, default="vicuna_v1")
parser.add_argument("--answer-with-sentence", type=bool, default=False)
parser.add_argument("--num-chunks", type=int, default=1)
parser.add_argument("--chunk-idx", type=int, default=0)
parser.add_argument("--temperature", type=float, default=0)
parser.add_argument("--top_p", type=float, default=None)
parser.add_argument("--num_beams", type=int, default=1)
parser.add_argument("--max_new_tokens", type=int, default=128)
parser.add_argument("--load-8bit", action="store_true")
parser.add_argument("--load-4bit", action="store_true")
parser.add_argument("--debug", action="store_true")
args = parser.parse_args()

disable_torch_init()

model_name = get_model_name_from_path(args.model_path)
tokenizer, model, image_processor, context_len = load_pretrained_model(args.model_path, args.model_base, model_name, args.load_8bit, args.load_4bit, device=args.device)

if "llama-2" in model_name.lower():
    conv_mode = "llava_llama_2"
elif "mistral" in model_name.lower():
    conv_mode = "mistral_instruct"
elif "v1.6-34b" in model_name.lower():
    conv_mode = "chatml_direct"
elif "v1" in model_name.lower():
    conv_mode = "llava_v1"
elif "mpt" in model_name.lower():
    conv_mode = "mpt"
else:
    conv_mode = "llava_v0"

In [ ]:
%env CUDA_VISIBLE_DEVICES=0
import os
os.environ["CUDA_VISIBLE_DEVICES"]="0"
if args.conv_mode is not None and conv_mode != args.conv_mode:
    print('[WARNING] the auto inferred conversation mode is {}, while `--conv-mode` is {}, using {}'.format(conv_mode, args.conv_mode, args.conv_mode))
else:
    args.conv_mode = conv_mode

conv = conv_templates[args.conv_mode].copy()
if "mpt" in model_name.lower():
    roles = ('user', 'assistant')
else:
    roles = conv.roles

IMAGE_PATH = "./apple.png"
image = load_image(IMAGE_PATH)
image_size = image.size
# Similar operation in model_worker.py
image_tensor = process_images([image], image_processor, model.config)
if type(image_tensor) is list:
    image_tensor = [image.to(model.device, dtype=torch.float16) for image in image_tensor]
else:
    image_tensor = image_tensor.to(model.device, dtype=torch.float16)


inp = "Is there any apple in the image?\nAnswer with a single word or short phrase."
print(f"{roles[1]}: ", end="")

if image is not None:
    # first message
    if model.config.mm_use_im_start_end:
        inp = DEFAULT_IM_START_TOKEN + DEFAULT_IMAGE_TOKEN + DEFAULT_IM_END_TOKEN + '\n' + inp
    else:
        inp = DEFAULT_IMAGE_TOKEN + '\n' + inp
    image = None

conv.append_message(conv.roles[0], inp)
conv.append_message(conv.roles[1], None)
prompt = conv.get_prompt()

input_ids = tokenizer_image_token(prompt, tokenizer, IMAGE_TOKEN_INDEX, return_tensors='pt').unsqueeze(0).to(model.device)
stop_str = conv.sep if conv.sep_style != SeparatorStyle.TWO else conv.sep2
keywords = [stop_str]
streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

with torch.inference_mode():
    output = model.generate(
        input_ids,
        images=image_tensor,
        image_sizes=[image_size],
        do_sample= False,
        temperature=0,
        max_new_tokens=args.max_new_tokens,
        streamer=streamer,
        output_hidden_states=True,
        output_attentions=True,
        output_scores=True,
        return_dict_in_generate=True  # Return as a dictionary so you can access attentions
    )

norm = model.model.norm
lm_head = model.lm_head
prefill_hidden_states = torch.stack(output.hidden_states[0])

In [ ]:
tokenizer.decode([1141,26673,30296])

In [ ]:
compared_word = 'igh'
black_embeddings = model.get_input_embeddings()(tokenizer(compared_word, return_tensors='pt').input_ids.to(model.device))[0,1:,:].mean(dim=0)
black_embeddings = model.get_input_embeddings()(torch.tensor(30296).to(model.device))

F.cosine_similarity(black_embeddings, prefill_hidden_states[0,0,35:611,:], dim=-1).topk(5)

In [ ]:
def find_most_similar_tokens(target_embedding, model, tokenizer, top_k=10):

    all_embeddings = model.get_input_embeddings().weight  # shape: [vocab_size, embedding_dim]
    
    if target_embedding.dim() == 1:
        target_embedding = target_embedding.unsqueeze(0)  # [1, embedding_dim]
    
    similarities = F.cosine_similarity(target_embedding, all_embeddings, dim=-1)  # [vocab_size]
    
    top_similarities, top_indices = similarities.topk(top_k)
    
    similar_tokens = []
    for idx in top_indices:
        token = tokenizer.decode([idx.item()])
        similar_tokens.append(token)
    
    return similar_tokens, top_similarities, top_indices

layer_idx = 0
target_emb = prefill_hidden_states[layer_idx,0,361+35,:]
similar_tokens, similarities, token_ids = find_most_similar_tokens(target_emb, model, tokenizer, top_k=10)

print("The most similar tokens:")
for i, (token, sim, token_id) in enumerate(zip(similar_tokens, similarities, token_ids)):
    print(f"{i+1}. Token: '{token}' (ID: {token_id.item()}) - Similarity: {sim.item():.4f}")